# v10-infer-submit — suy luận 1.000 câu test, xuất submission BTC

Checkpoint: `tot_nhat/` của phiên 3 (train trên prompt **có số hiệu**, **không trần ký tự**).

Ba assert ở cell 1 là ba cách sai im lặng đã gặp thật:
`LEGALQA_SO_HIEU` sai → prompt lệch phân phối train; `MAX_CHARS` còn trần → prompt ngắn
hơn lúc train; glob bắt `phien2` → nộp bằng checkpoint cũ.

Nghiệm thu ở cell 4 kiểm **đủ 1.000 khoá và không khoá nào rỗng**: `eval_qa` của BTC
ném nếu số khoá lệch, nên thiếu một khoá là 0 điểm toàn tập chứ không phải mất một câu.

Cần gắn: GPU T4 x2, Internet ON, `official`, `legalqa-v10-src`, `legalqa-clean`,
`legalqa-sft-phien3`.

In [ ]:
import os, sys, shutil, zipfile, subprocess, json, gc
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())

!pip -q install transformers==5.14.1 peft nltk
!pip -q uninstall -y torchao

# Adapter phiên 3 được train trên prompt CÓ số hiệu và KHÔNG trần ký tự. Cả hai phải
# khớp, không phải một trong hai: sai bất kỳ cái nào là đưa model vào phân phối nó chưa
# từng thấy, và kết quả tệ đi mà không có dấu vết nào trong log.
os.environ["LEGALQA_SO_HIEU"] = "1"
from legalqa.prompt import CO_SO_HIEU
from build_sft import TOP_N, MAX_CHARS
assert CO_SO_HIEU is True, "prompt đang TẮT số hiệu — sai checkpoint phiên 3"
assert TOP_N == 5 and MAX_CHARS > 10**6, f"codebase CŨ: top_n={TOP_N} max_chars={MAX_CHARS}"
assert (REPO / "scripts/infer.py").exists()
print(f"số hiệu ON | top_n={TOP_N} max_chars={MAX_CHARS:,} (không trần)")

In [ ]:
ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)
for n in ("chunks.jsonl", "bm25.pkl", "corpus.jsonl"):
    assert (CLEAN / n).exists(), f"thiếu {n} trong legalqa-clean"

# Tập test BTC: dict {id: {question, answer=None}}, KHÔNG phải jsonl của ta.
# `nap_cau_hoi()` nhận cả hai dạng theo đuôi file nên đường dẫn phải giữ đuôi .json.
CAU_HOI = next(Path("/kaggle/input").glob("**/public-official.json"), None)
assert CAU_HOI, "Không thấy public-official.json — gắn dataset official"
_d = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert len(_d) == 1000, f"tập test có {len(_d)} câu, phải là 1.000"
assert all(v.get("answer") is None for v in _d.values()), "tập test có đáp án?"
print(CAU_HOI, len(_d), "câu")

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/kaggle/working/base")
    FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX
    print("vá tên shard ->", BASE)

# Adapter phiên 3. Lọc theo tên dataset: gắn cả phien2 lẫn phien3 thì glob bắt trúng
# phien2 trước và ta suy luận bằng checkpoint cũ mà không hay.
ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")
                if "phien3" in str(q)), None)
assert ADAPTER, "không thấy tot_nhat/ của phiên 3 — gắn dataset legalqa-sft-phien3"

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    RERANK = str(local[0])
else:
    from huggingface_hub import snapshot_download
    RERANK = snapshot_download("AITeamVN/Vietnamese_Reranker")
print("base   ", BASE)
print("adapter", ADAPTER)
print("rerank ", RERANK)

In [ ]:
# NỐI TIẾP lượt trước. v1 làm xong pha 1 (1.000 prompt, 1,35 s/câu) và 840/1.000 câu
# pha 2 rồi hết `--gio-pha2 5.5`: sinh thật tốn 23,6 s/câu, không phải 17 s/câu như ước
# ở trên — 1.000 câu cần ~6,6 giờ chỉ riêng pha 2.
#
# `da_co()` trong infer.py lọc theo id đã có trong file ĐÍCH, nên chỉ cần đặt hai file
# của lượt trước vào `--ra` là 160 câu còn lại được sinh tiếp. Không nối thì trả lại 6
# giờ GPU để sinh lại đúng 840 câu đã có sẵn.
#
# COPY chứ không symlink: infer.py mở hai file này ở chế độ APPEND, mà /kaggle/input là
# read-only — symlink thì ghi vào là nổ, và nổ sau khi đã nạp xong model.
RA = Path("/kaggle/working/out"); RA.mkdir(parents=True, exist_ok=True)
for ten in ("prompts.jsonl", "du_doan.jsonl"):
    # Glob KHÔNG có "out/": Kaggle tự bung dataset zip và làm PHẲNG cấu trúc — đẩy
    # out/du_doan.jsonl lên thì `datasets files` trả về "du_doan.jsonl" ở gốc. Bỏ tiền
    # tố thì bắt được cả dạng dataset lẫn dạng kernel_sources (.../out/du_doan.jsonl).
    cu = next((q for q in Path("/kaggle/input").glob(f"**/{ten}")), None)
    if cu:
        shutil.copy(cu, RA / ten)
        print(f"nối {ten}: {sum(1 for _ in open(RA / ten, encoding='utf-8')):,} dòng")
    else:
        print(f"KHÔNG thấy {ten} của lượt trước — pha tương ứng chạy lại từ đầu")

# Ngân sách: retrieval ~2s/câu (33 phút cho 1.000) + sinh ~17s/câu (prompt dài hơn
# 500-câu-bỏ-qua vì bỏ trần ký tự) = ~4,7 giờ. Hai mốc giờ tách riêng để pha 1 không ăn
# mất giờ của pha 2; cả hai pha ghi tăng dần ra jsonl nên chạy lại kernel là tiếp được.
#
# --submission ghi submission.json + submission.zip với ĐỦ 1.000 khoá, câu chưa sinh
# được để chuỗi rỗng. `eval_qa` của BTC ném nếu số khoá lệch, nên thiếu khoá là 0 điểm
# toàn tập, không phải mất một câu.
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI),
    "--model", str(BASE), "--adapter", str(ADAPTER), "--rerank", RERANK,
    "--ra", "/kaggle/working/out", "--submission",
    "--batch-size", "32", "--pool", "50",
    "--max-new", "1024", "--gen-bs", "2",
    "--gio-pha1", "1.0", "--gio-pha2", "8.5",
], check=True)

In [ ]:
# Không có gold nên không chấm được. Nghiệm thu đúng thứ quyết định điểm nộp: HÌNH
# DẠNG submission, rồi mới đến chẩn đoán bịa (so với b318 để biết fix số hiệu có ăn).
import re
from collections import Counter

sub = json.loads(Path("/kaggle/working/out/submission.json").read_text(encoding="utf-8"))
goc = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert set(sub) == set(goc), f"khoá lệch: {len(set(goc) - set(sub))} thiếu, {len(set(sub) - set(goc))} lạ"
assert all(isinstance(v, dict) and isinstance(v.get("answer"), str) for v in sub.values())
rong = [k for k, v in sub.items() if not v["answer"].strip()]
print(f"submission: {len(sub)} khoá | rỗng {len(rong)}")
assert not rong, f"{len(rong)} câu chưa sinh được — tăng --gio-pha2 rồi chạy lại kernel"

rows = [json.loads(l) for l in open("/kaggle/working/out/du_doan.jsonl", encoding="utf-8")]
pr = {json.loads(l)["id"]: json.loads(l)["prompt"]
      for l in open("/kaggle/working/out/prompts.jsonl", encoding="utf-8")}
NAM = re.compile(r"\b(?:19|20)\d{2}\b")
DIEU = re.compile(r"Điều\s+(\d+[a-zA-Z]?)")
st = Counter()
for x in rows:
    p, c = str(x["du_doan"]), pr[x["id"]]
    pn, cn = set(NAM.findall(p)), set(NAM.findall(c))
    pd, cd = set(DIEU.findall(p)), set(DIEU.findall(c))
    st["n"] += 1
    # Không có gold nên KHÔNG đo được `nam_bia_SAI`. Chỉ còn `nam_bia` = năm trong dự
    # đoán mà prompt không có. Ghi rõ để đừng ai so con số này với 46,7% của dev.
    st["nam_bia"] += bool(pn - cn)
    st["dieu_bia"] += bool(pd - cd)
    st["cut"] += not p.rstrip().endswith((".", "!", "?", '"', ":", ";"))
    st["prompt_co_nam"] += bool(cn)
n = st["n"]
print(f"\ntrên {n} câu test:")
for k in ("nam_bia", "dieu_bia", "cut", "prompt_co_nam"):
    print(f"  {k:14} {st[k]:4}  {st[k]/n:6.1%}")
print("\nb318 (CHƯA có số hiệu) trên 500 câu bỏ qua: nam_bia 62,6%  dieu_bia 10,4%  cut 10,8%")
print("b318 trên 120 câu dev:                        nam_bia 65,0%  dieu_bia  3,3%  cut  6,7%")
print("prompt_co_nam là mẫu số thật: dev cũ chỉ 56,7% prompt có năm nào.")